# Proyecto: Predicción de cancelación de clientes en Beta Bank

## Objetivo general

Beta Bank está perdiendo clientes poco a poco cada mes, y al banco le sale más barato retener a un cliente que conseguir uno nuevo. Por eso necesita anticiparse y detectar quién está por irse antes de que ocurra.

En este proyecto construimos un modelo de clasificación que, a partir del comportamiento pasado de cada cliente, prediga si va a abandonar el banco. El requisito es alcanzar un **F1 de al menos 0.59 en el conjunto de prueba**. Además vamos a medir el **AUC-ROC** y comparar ambas métricas.

### Descripción de los datos

El archivo `Churn.csv` contiene 10 000 registros de clientes:

| Columna | Descripción |
|---|---|
| `RowNumber` | índice de la fila |
| `CustomerId` | identificador único del cliente |
| `Surname` | apellido |
| `CreditScore` | valor de crédito |
| `Geography` | país de residencia |
| `Gender` | sexo |
| `Age` | edad |
| `Tenure` | años de maduración del depósito a plazo fijo |
| `Balance` | saldo de la cuenta |
| `NumOfProducts` | número de productos bancarios contratados |
| `HasCrCard` | tiene tarjeta de crédito (1 sí / 0 no) |
| `IsActiveMember` | cliente activo (1 sí / 0 no) |
| `EstimatedSalary` | salario estimado |
| `Exited` | **objetivo**: el cliente se fue (1 sí / 0 no) |

## Paso 1: Preparación y exploración de los datos

**Objetivo:** revisar la calidad de los datos (tipos, valores ausentes, duplicados) y dejar el dataset listo para entrenar: todas las columnas numéricas, sin ausentes y solo con variables que digan algo del comportamiento del cliente.

In [13]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier  # modelo que vamos a entrenar
from sklearn.model_selection import train_test_split  # para dividir el dataset
from sklearn.metrics import (  # métricas para evaluar la clasificación
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# cargamos el dataset de clientes del banco
df = pd.read_csv('Churn.csv')

In [14]:
# exploración inicial: estructura, contenido y estadísticas del dataset
df.info()  # tipos de dato y conteo de valores no nulos por columna
print()
print(df.head())  # primeras filas para inspeccionar el formato
print()
print(df.describe())  # estadísticas descriptivas de las columnas numéricas

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.2 MB

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave

In [15]:
# revisamos la calidad de los datos: ausentes y duplicados
print('Valores ausentes por columna:')
print(df.isnull().sum())
print()
print(f'Filas duplicadas: {df.duplicated().sum()}')
# CustomerId identifica de forma única a cada cliente: si se repitiera, habría clientes cargados dos veces
print(f'CustomerId duplicados: {df["CustomerId"].duplicated().sum()}')
print()
# categorías de las columnas de texto, para decidir cómo codificarlas más adelante
print(df['Geography'].value_counts())
print()
print(df['Gender'].value_counts())

Valores ausentes por columna:
RowNumber            0
CustomerId           0
Surname              0
CreditScore          0
Geography            0
Gender               0
Age                  0
Tenure             909
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Exited               0
dtype: int64

Filas duplicadas: 0
CustomerId duplicados: 0

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Gender
Male      5457
Female    4543
Name: count, dtype: int64


### Valores ausentes en `Tenure`

`Tenure` es la única columna con datos faltantes, y por eso aparece como decimal y no como entero.

Antes de rellenarla queremos saber por qué faltan esos datos: si los clientes sin `Tenure` se comportaran distinto al resto, la ausencia sería información en sí misma y taparla con un solo valor introduciría un sesgo. Comparamos los dos grupos antes de decidir.

In [16]:
# máscara que separa las filas con Tenure ausente de las completas
tenure_ausente = df['Tenure'].isnull()

print(f'Filas con Tenure ausente: {tenure_ausente.sum()} ({tenure_ausente.mean():.2%})')
print()
# si la tasa de cancelación difiriera mucho entre ambos grupos, la ausencia estaría ligada al objetivo
print('Tasa de cancelación según Tenure ausente:')
print(df.groupby(tenure_ausente)['Exited'].mean())
print()
print('Distribución de Geography según Tenure ausente:')
print(df.groupby(tenure_ausente)['Geography'].value_counts(normalize=True).unstack())
print()
print('Perfil numérico promedio según Tenure ausente:')
print(df.groupby(tenure_ausente)[['Age', 'CreditScore', 'Balance', 'EstimatedSalary']].mean())

Filas con Tenure ausente: 909 (9.09%)

Tasa de cancelación según Tenure ausente:
Tenure
False    0.203938
True     0.201320
Name: Exited, dtype: float64

Distribución de Geography según Tenure ausente:
Geography    France   Germany     Spain
Tenure                                 
False      0.500495  0.252227  0.247278
True       0.510451  0.237624  0.251925

Perfil numérico promedio según Tenure ausente:
              Age  CreditScore       Balance  EstimatedSalary
Tenure                                                       
False   38.949181   650.736553  76522.740015    100181.214924
True    38.647965   648.451045  76117.341474     99180.389373


Los dos grupos son prácticamente iguales: misma tasa de cancelación, misma distribución por país y los mismos promedios de edad, crédito, saldo y salario. La ausencia no está ligada a ninguna característica del cliente ni al objetivo, así que podemos rellenarla sin distorsionar los datos.

Elegimos la mediana en lugar de la media porque no se ve afectada por valores extremos y devuelve un número entero de años, que es como se mide la variable (la media daría un decimal).

In [17]:
# rellenamos los ausentes con la mediana y devolvemos la columna a entero (años completos)
print(f'Mediana de Tenure: {df["Tenure"].median()}')
df['Tenure'] = df['Tenure'].fillna(df['Tenure'].median()).astype(int)

Mediana de Tenure: 5.0


### Columnas descartadas y codificación de las categóricas

`RowNumber` es solo un contador de filas y `CustomerId` un identificador arbitrario: ninguno dice nada sobre el comportamiento del cliente. `Surname` tampoco, y además tiene miles de valores distintos, así que al codificarlo generaría un montón de columnas y el modelo terminaría memorizando apellidos en vez de aprender patrones. Descartamos las tres.

`Geography` y `Gender` sí aportan información, pero son texto y scikit-learn solo trabaja con números. Las pasamos a One-Hot Encoding con `drop_first=True`: con tres países bastan dos columnas, porque la tercera queda implícita. No usamos una codificación tipo France=0, Germany=1, Spain=2 porque inventaría un orden entre países que no existe.

In [18]:
# eliminamos las columnas identificadoras, que no aportan información predictiva
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# codificamos las variables categóricas con One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [19]:
# verificación final: ninguna columna de texto y ningún valor ausente
df.info()
print()
print(df.head())
print()
print(f'Valores ausentes totales: {df.isnull().sum().sum()}')

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        10000 non-null  int64  
 1   Age                10000 non-null  int64  
 2   Tenure             10000 non-null  int64  
 3   Balance            10000 non-null  float64
 4   NumOfProducts      10000 non-null  int64  
 5   HasCrCard          10000 non-null  int64  
 6   IsActiveMember     10000 non-null  int64  
 7   EstimatedSalary    10000 non-null  float64
 8   Exited             10000 non-null  int64  
 9   Geography_Germany  10000 non-null  bool   
 10  Geography_Spain    10000 non-null  bool   
 11  Gender_Male        10000 non-null  bool   
dtypes: bool(3), float64(2), int64(7)
memory usage: 732.6 KB

   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619   42       2       0.00              1          1   
1          608   41       1   838

**Conclusión:** el dataset traía 10 000 registros y 14 columnas, sin filas duplicadas ni `CustomerId` repetidos.

La única columna con ausentes era `Tenure`: 909 valores, un 9.09 % del total. Las filas afectadas tienen la misma tasa de cancelación que el resto (20.1 % frente a 20.4 %) y el mismo perfil de país, edad, crédito, saldo y salario, así que las rellenamos con la mediana (5 años) y devolvimos la columna a entero.

Después de descartar las tres columnas identificadoras y codificar `Geography` y `Gender`, quedamos con **10 000 filas y 12 columnas**: 11 características y el objetivo `Exited`. Sin valores ausentes y con todo en formato numérico, los datos ya están listos para entrenar.

## Paso 2: Examen del equilibrio de clases y modelo base

**Objetivo:** medir qué tan desequilibradas están las clases, dividir los datos en entrenamiento, validación y prueba, y entrenar un primer modelo sin corregir ese desequilibrio para tener una referencia contra la cual comparar las mejoras del paso 3.

In [20]:
# trabajamos sobre una copia para no alterar el DataFrame original
df_modelo = df.copy()

# separamos las características de la columna objetivo
features = df_modelo.drop('Exited', axis=1)
target = df_modelo['Exited']

# proporción de clases, en número y en porcentaje
print(target.value_counts())
print()
print(target.value_counts(normalize=True) * 100)

Exited
0    7963
1    2037
Name: count, dtype: int64

Exited
0    79.63
1    20.37
Name: proportion, dtype: float64


Las clases están desequilibradas: casi el 80 % de los clientes se queda y solo el 20 % se va, 1 de cada 5. Con esa proporción, un modelo que dijera siempre "se queda" acertaría el 80 % de las veces sin haber aprendido nada, así que la exactitud no nos sirve como métrica principal.

### División en entrenamiento, validación y prueba

Repartimos los datos en 60 % entrenamiento, 20 % validación y 20 % prueba. Usamos `stratify` para que los tres conjuntos conserven la misma proporción de clientes que se van: siendo la clase minoritaria solo el 20 %, una división al azar puede dejar proporciones distintas en cada conjunto y la comparación entre modelos dejaría de ser justa.

El conjunto de validación es el que usaremos para elegir el mejor modelo. El de prueba queda reservado y no lo tocamos hasta el paso 4.

In [21]:
# primer corte: 60% para entrenamiento y 40% que repartiremos entre validación y prueba
features_train, features_rest, target_train, target_rest = train_test_split(
    features, target, test_size=0.4, random_state=12345, stratify=target
)

# segundo corte: partimos ese 40% por la mitad → 20% validación y 20% prueba
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest, target_rest, test_size=0.5, random_state=12345, stratify=target_rest
)

# comprobamos los tamaños y que la proporción de clientes que se van se haya conservado
for nombre, conjunto in [('Entrenamiento', target_train), ('Validación', target_valid), ('Prueba', target_test)]:
    print(f'{nombre}: {len(conjunto)} filas, {conjunto.mean():.2%} de clientes que se van')

Entrenamiento: 6000 filas, 20.37% de clientes que se van
Validación: 2000 filas, 20.40% de clientes que se van
Prueba: 2000 filas, 20.35% de clientes que se van


### Modelo base sin corregir el desequilibrio

Entrenamos un bosque aleatorio con los hiperparámetros por defecto y sin tocar el desequilibrio, para tener una referencia de partida. El ajuste de hiperparámetros y las técnicas de balanceo llegan en el paso 3.

Definimos primero una función que calcule todas las métricas de golpe, así la reutilizamos con cada modelo que entrenemos.

In [22]:
# función para calcular de una sola vez todas las métricas y poder comparar modelos después
def evaluar_modelo(nombre, modelo, features, target):
    predicciones = modelo.predict(features)
    # el AUC-ROC se calcula con la probabilidad de la clase 1, no con la etiqueta predicha
    probabilidades = modelo.predict_proba(features)[:, 1]

    return {
        'modelo': nombre,
        'exactitud': accuracy_score(target, predicciones),
        'precisión': precision_score(target, predicciones),
        'recall': recall_score(target, predicciones),
        'F1': f1_score(target, predicciones),
        'AUC-ROC': roc_auc_score(target, probabilidades),
    }

In [23]:
# entrenamos el modelo base con los hiperparámetros por defecto, sin corregir el desequilibrio
modelo_base = RandomForestClassifier(random_state=12345)
modelo_base.fit(features_train, target_train)

resultado_base = evaluar_modelo('Bosque aleatorio (sin balancear)', modelo_base, features_valid, target_valid)
pd.DataFrame([resultado_base])

,modelo,exactitud,precisión,recall,F1,AUC-ROC
0,Bosque aleatorio (sin balancear),0.868,0.768657,0.504902,0.609467,0.862824


In [24]:
# la matriz de confusión nos deja ver cuántos clientes que sí se fueron el modelo dejó pasar
print('Matriz de confusión (validación):')
print(confusion_matrix(target_valid, modelo_base.predict(features_valid)))

Matriz de confusión (validación):
[[1530   62]
 [ 202  206]]


Para dimensionar el resultado lo comparamos contra un modelo constante que predice que ningún cliente se va. Es el peor modelo posible desde el punto de vista del negocio, y sirve para ver cuánta exactitud se puede conseguir sin aprender nada.

In [25]:
# modelo constante que predice que ningún cliente se va, como punto de comparación
predicciones_constantes = pd.Series(0, index=target_valid.index)

print(f'Exactitud: {accuracy_score(target_valid, predicciones_constantes):.4f}')
print(f'Recall: {recall_score(target_valid, predicciones_constantes):.4f}')
print(f'F1: {f1_score(target_valid, predicciones_constantes):.4f}')

Exactitud: 0.7960
Recall: 0.0000
F1: 0.0000


**Conclusión:** las clases están desequilibradas en una proporción cercana al 80/20: 7 963 clientes se quedaron y 2 037 se fueron, es decir 1 de cada 5.

El bosque aleatorio entrenado sin corregir ese desequilibrio alcanza una exactitud del 86.8 % y una precisión de 0.77, pero su recall es de apenas 0.50. Mirando la matriz de confusión: de los 408 clientes del conjunto de validación que sí se fueron, el modelo detectó 206 y dejó pasar 202. Cuando dice que un cliente se va suele acertar, pero se le escapa la mitad de los que realmente se van, que es justo el grupo que al banco le interesa retener.

El F1 quedó en 0.6095 y el AUC-ROC en 0.8628. Llama la atención que el F1 ya supere el umbral de 0.59 sin haber corregido nada: el bosque aleatorio tolera bastante bien el desequilibrio. Aun así este número es sobre validación, no sobre prueba, y con la mitad de los clientes en riesgo sin detectar hay margen claro de mejora en el paso 3.

Como referencia, un modelo que predijera que ningún cliente se va obtendría un 79.6 % de exactitud con un F1 de 0. Eso confirma que la exactitud no sirve para medir este problema y que las métricas a seguir son el F1 y el recall.